<a href="https://colab.research.google.com/github/Annie00000/Project/blob/main/3_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 粗略版本 (type = bin)

要先將 DataFrame 的格式從矩陣型態（Wide Format）轉換為座標清單型態（Long Format），過濾掉不需要的點後，再使用 plotly.graph_objects (go) 進行繪圖

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# 1. 模擬資料環境 (建立一個符合您描述的 DataFrame)
rows = list(range(-30, 31))  # y 座標 -30 到 30
cols = list(range(80))       # x 座標 0 到 79
df = pd.DataFrame('', index=rows, columns=cols)

# 隨機填入一些測試資料 (1, 28, 37 以及 0 或 '')
df.loc[0, 10] = 1
df.loc[-15, 20] = 28
df.loc[25, 45] = 37
df.loc[5, 5] = 0   # 這是 0，應該被過濾
df.loc[-5, 5] = '' # 這是空字串，應該被過濾

# 2. 資料轉換與過濾
# 將 DataFrame 攤平，轉換成 x, y, value 的形式
df_stacked = df.stack().reset_index()
df_stacked.columns = ['y', 'x', 'value']

# 強制轉換 value 型別以利判斷 (處理混和型態)
def is_valid(val):
    if val == '' or val == 0 or val == '0':
        return False
    return True

# 篩選出非 '' 且非 0 的點
mask = df_stacked['value'].apply(is_valid)
plot_df = df_stacked[mask].copy()

# 3. 使用 Plotly 繪製互動式圖表
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_df['x'],
    y=plot_df['y'],
    mode='markers',
    marker=dict(
        color='rgb(0,0,255)', # 設定為藍色
        size=10,
        line=dict(width=1, color='DarkSlateGrey')
    ),
    text=plot_df['value'],    # 滑鼠懸停時顯示的內容
    name='目標點位',           # Legend 標示名稱
    hovertemplate='<b>座標</b>: (%{x}, %{y})<br><b>數值</b>: %{text}<extra></extra>'
))

# 4. 圖表佈局設定
fig.update_layout(
    title='DataFrame 座標點位圖',
    xaxis=dict(title='X 座標 (Col Index)', range=[-1, 80], dtick=10),
    yaxis=dict(title='Y 座標 (Row Index)', range=[-31, 31], dtick=10),
    clickmode='event+select', # 啟用點擊事件模式
    hovermode='closest'
)

# 顯示圖表
fig.show()



因為我們已經過濾掉了 '' 與 0 的點，畫布上不存在這些座標的物件，所以使用者無法點擊到它們。

如果您是在網頁環境（如 Dash 或 Jupyter）中使用，當點擊這些點時，Plotly 會回傳該點的 curveNumber, pointNumber, x, y 等資訊。

### 2. type = bin /defect_pattern

In [ ]:
import pandas as pd
import plotly.graph_objects as go

def plot_interactive_map(df_bin, df_defect=None, mode='bin'):
    """
    df_bin: 原始資料 (index: -30~30, cols: 0-79)
    df_defect: 包含 x, y, defect_pattern 的 DataFrame (選填)
    mode: 'bin' 或 'defect'
    """
    fig = go.Figure()

    # 1. 資料預處理：轉為 Long Format (將矩陣轉為座標清單並過濾 0/'')
    df_stacked = df_bin.stack().reset_index()
    df_stacked.columns = ['y', 'x', 'value']
    # 只保留有效的點 (非空、非零)
    main_points = df_stacked[~df_stacked['value'].isin(['', 0, '0'])].copy()

    # --- 畫法 A: Bin 模式 ---
    if mode == 'bin':
        fig.add_trace(go.Scatter(
            x=main_points['x'],
            y=main_points['y'],
            mode='markers',
            marker=dict(color='rgb(0,0,255)', size=8),
            text=main_points['value'],
            name='Bin Data (1/9/28/37...)',
            hovertemplate='<b>Bin</b>: %{text}<br>座標: (%{x}, %{y})<extra></extra>'
        ))

    # --- 畫法 B: Defect 模式 ---
    elif mode == 'defect' and df_defect is not None:
        # 將 defect 資訊合併到 main_points 中 (以座標 x, y 為基準)
        # 確保 df_defect 的欄位是 ['x', 'y', 'defect_pattern']
        merged = pd.merge(main_points, df_defect, on=['x', 'y'], how='left')

        # 排除 defect_pattern 為 0 或 '' 的無效資訊
        merged['defect_pattern'] = merged['defect_pattern'].replace(['', 0, '0'], None)

        # 分成兩群：有 Pattern vs 無 Pattern
        has_pattern = merged[merged['defect_pattern'].notna()]
        no_pattern = merged[merged['defect_pattern'].isna()]

        # 1. 無 Pattern 的點 -> 畫黑點
        fig.add_trace(go.Scatter(
            x=no_pattern['x'],
            y=no_pattern['y'],
            mode='markers',
            marker=dict(color='black', size=6, opacity=0.5),
            name='Normal (No Pattern)',
            hoverinfo='skip' # 黑點通常不需特別標註，減少干擾
        ))

        # 2. 有 Pattern 的點 -> 標註 Legend
        # 如果你想讓不同的 pattern 有不同顏色，這裡可以再細分，
        # 否則就統一標註為 Defect
        fig.add_trace(go.Scatter(
            x=has_pattern['x'],
            y=has_pattern['y'],
            mode='markers',
            marker=dict(
                color='rgb(255, 0, 0)',
                size=10,
                symbol='diamond',
                line=dict(width=1, color='white')
            ),
            text=has_pattern['defect_pattern'],
            name='Defect Pattern',
            hovertemplate='<b>Defect</b>: %{text}<br>座標: (%{x}, %{y})<extra></extra>'
        ))

    # --- 通用佈局設定 ---
    fig.update_layout(
        title=f'Interactive Map - {mode.upper()} Mode',
        xaxis=dict(title='X (Column)', range=[-1, 80], gridcolor='lightgrey', dtick=10),
        yaxis=dict(title='Y (Row)', range=[-31, 31], gridcolor='lightgrey', dtick=10),
        plot_bgcolor='white',
        clickmode='event+select'
    )

    return fig

# --- 使用範例 ---

# 假設 df_bin 已經存在
# 假設 df_defect 格式如下:
# df_defect = pd.DataFrame({'x': [10, 20], 'y': [0, -15], 'defect_pattern': ['Scratch', 'Crack']})

# 畫 Bin 圖
# fig_bin = plot_interactive_map(df_bin, mode='bin')
# fig_bin.show()

# 畫 Defect 圖
# fig_defect = plot_interactive_map(df_bin, df_defect, mode='defect')
# fig_defect.show()

### 3.綠色外框

In [ ]:
    # --- 分離出 0.5 的點與其他的點 ---
    points_05 = valid_points[valid_points['value_num'] == 0.5]
    points_others = valid_points[valid_points['value_num'] != 0.5]

    # --- 核心畫法：0.5 的綠色點 (不論模式都先畫，且無 hover) ---
    fig.add_trace(go.Scatter(
        x=points_05['x'],
        y=points_05['y'],
        mode='markers',
        marker=dict(color='green', size=6, opacity=0.6),
        name='Reference (0.5)',
        hoverinfo='skip'  # 隱藏所有 hover 資訊
    ))

----

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# --- 1. 準備模擬資料 (延用前述環境) ---
rows = list(range(-30, 31))
cols = list(range(80))
df = pd.DataFrame('', index=rows, columns=cols)
df.loc[0, 10] = 1
df.loc[-15, 20] = 28
df.loc[25, 45] = 37
df.loc[10, 60] = 0.5 # 新增的參考點

# 1.1 資料預處理 (Long Format)
df_stacked = df.stack().reset_index()
df_stacked.columns = ['y', 'x', 'value']
df_stacked['value_num'] = pd.to_numeric(df_stacked['value'], errors='coerce')

# 1.2 找出所有「有效點」(非 0/'' 的點，包含 0.5)
# 用這些點來計算最外圍邊界
all_points = df_stacked[
    (df_stacked['value'] != '') &
    (df_stacked['value_num'] != 0)
].copy()

# --- 2. 計算外接圓參數 ---

# 2.1 計算資料點的極值
x_min, x_max = all_points['x'].min(), all_points['x'].max()
y_min, y_max = all_points['y'].min(), all_points['y'].max()

# 2.2 計算幾何中心
center_x = (x_min + x_max) / 2
center_y = (y_min + y_max) / 2

# 2.3 計算從中心到最遠點的距離 (數據單位)
# 這是為了確保圓圈能包住所有點
max_dist_data = np.sqrt(((all_points['x'] - center_x)**2 +
                          (all_points['y'] - center_y)**2).max())

# --- 3. 物理單位轉數據單位的估算 (關鍵步) ---
# 這是一個估算值。在鎖定 1:1 比例下：
# 我們假設 1 個數據單位 (例如 X 軸 0 到 1) 約為 0.2 公分 (具體取決於螢幕)
# 所以，2 公分約等同於 10 個數據單位。
UNIT_PER_CM_ESTIMATE = 5 # 假設 1 cm = 5 個數據單位
EXTRA_WIDTH_DATA = 2 * UNIT_PER_CM_ESTIMATE # 2 cm 等效的數據距離

# 2.4 最終圓形半徑 (數據單位)
final_radius = max_dist_data + EXTRA_WIDTH_DATA

# --- 4. 繪製圖表 ---
fig = go.Figure()

# 4.1 繪製點 (使用您原本的 logic，這裡簡化為統一 Trace)
fig.add_trace(go.Scatter(
    x=all_points['x'],
    y=all_points['y'],
    mode='markers',
    marker=dict(color='blue', size=8),
    name='Data Points'
))

# 4.2 **加入綠圓圈 (Shape)**
# shape 的座標是相對於數據軸的
fig.add_shape(
    type="circle",
    xref="x", yref="y",
    x0=center_x - final_radius, # 圓形左邊界
    y0=center_y - final_radius, # 圓形下邊界
    x1=center_x + final_radius, # 圓形右邊界
    y1=center_y + final_radius, # 圓形上邊界
    line=dict(
        color="limegreen", # 綠色
        width=3,          # 線寬
    ),
    fillcolor="rgba(0, 255, 0, 0.05)", # 輕微的綠色填充 (選填)
)

# --- 5. 佈局設定 (至關重要) ---
fig.update_layout(
    title='Map with 2cm Margin Green Circle',
    xaxis=dict(
        title='X (Column)',
        showgrid=True,
        zeroline=False,
    ),
    yaxis=dict(
        title='Y (Row)',
        showgrid=True,
        # **A. 鎖定比例 (1:1)**: 確保 X 和 Y 軸的單位長度在螢幕上看起來完全相同
        scaleanchor="x",
        scaleratio=1,
    ),
    plot_bgcolor='white',
    width=800, # 限制圖表物理寬度，有助於 1:1 的視覺確認
    height=700
)

fig.show()

**實作詳解：**

**1. 鎖定軸比例 (scaleanchor="x", scaleratio=1)：**
這在 fig.update_layout 的 yaxis 中設定。如果沒有這一步，Plotly 會自動拉伸座標軸以填滿視窗，這樣您畫出的「圓形 shape」看起來就會變成「橢圓形」，且「2公分」的物理距離也會隨之變形。鎖定 1:1 確保了數據單位的幾何誠實性。

**2. go.layout.Shape (Type: "circle")：**
我們不在 Scatter 中畫點，而是使用 Shape。這是一個幾何物件，我們提供它的外接正方形坐標 (x0, y0, x1, y1)，它就會自動生成圓形。

**3. 物理公分轉數據單位 (EXTRA_WIDTH_DATA)：**
由於 Plotly 不知道螢幕的 DPI，我們必須做一個假設性的轉換。在程式碼中，我估計「1公分」大約對應「5個數據單位」（這個值可能需要根據您實際顯示器的大小進行微調）。我們計算出外接圓半徑後，再加上這個「2公分等效數據距離」，得到最終的圓形。

這樣做出來的圖表，最外圍會有一個綠色的圓圈，包覆著所有點，且圓圈邊緣到資料點最遠處的視覺距離，大約就是您要求的「2公分」。

* **綠色外框可以考慮 : 按照上面這方法加上 or 原本方法在df周圍加上框**